# Codebase Maps for LLM Bug Localisation — Compiled Results Analysis

This notebook analyses results from an experiment testing whether providing an LLM coding agent
with a **codebase map** improves its ability to localise the file(s) that need to change to
resolve a GitHub issue, and whether that effect depends on **codebase size** and **model**.

**Study design**: for each of 45 selected issues (drawn from 15 open-source Python codebases,
stratified by codebase size and by whether the issue's ground truth is a single file, multiple
files, or an unconstrained mix), each of 4 models attempted file localisation under 4 map
conditions, repeated across multiple reps.

| `map_condition` | Description |
|---|---|
| `baseline` | No map — agent explores the repo directly with `list_files`/`read_file`/`search` |
| `structural` | AST-derived compact map: every class/function signature in the resolved package, no docstrings, capped at 55k tokens |
| `temporal_frequency` | Git edit-frequency map: how often each file has historically changed, capped at 55k tokens |
| `temporal_cochange` | Git co-change map: which files tend to change together in the same commit, capped at 55k tokens |

**Models** (confirmed complete trial sets only; see Step 0):
`mistral/ministral-3b-latest`, `deepseek/deepseek-v4-flash`,
`fireworks_ai/.../gpt-oss-120b`, `deepinfra/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B`.

This notebook is written to also serve as methodology documentation (referenced in the
dissertation), not just as runnable code — each step has a markdown cell explaining its purpose
and any judgement calls before the code that implements it. Places where a Python library
limitation forces an approximation relative to what's possible in R are flagged explicitly
rather than silently worked around.


## Setup

Required packages beyond the scientific-Python core (`pandas`, `numpy`, `matplotlib`):
`seaborn` (plotting), `statsmodels` (all formal modelling in Steps 4–7). Install with:

```bash
pip install seaborn statsmodels
```

**On R / `lme4`**: this environment has neither R nor `rpy2` installed. `lme4::glmer` is the
more mature tool for GLMMs with *crossed* random effects (`(1|issue) + (1|codebase)`) and
supports proper maximum-likelihood fitting with clean likelihood-ratio tests via `anova()`.
Python's `statsmodels` only offers `BinomialBayesMixedGLM`, a **variational Bayes**
approximation — not full maximum likelihood — for binomial mixed models with multiple
variance components. This is used here as the best available option, but Step 4 flags exactly
where this substitutes for what `lme4::glmer` + `anova()` would give you, and a marginal
(non-mixed) logistic regression is fit alongside it specifically to recover a valid frequentist
LRT for the fixed-effects structure (at the cost of not modelling the random-effects variance).
If a rigorous mixed-model LRT is required for the dissertation itself, re-running Step 4–6 in R
via `rpy2` or a standalone R script against `compiled_results.pkl` (readable from R via
`pandas`-exported CSV, or `reticulate`) is the recommended path — noted again inline at that
point.


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAVE_SEABORN = True
except ImportError:
    HAVE_SEABORN = False
    print("seaborn not installed -- some plots will fall back to plain matplotlib, "
          "or be skipped with a warning if they specifically need seaborn's logistic "
          "regplot. Install with `pip install seaborn`.")

try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    HAVE_STATSMODELS = True
except ImportError:
    HAVE_STATSMODELS = False
    print("statsmodels not installed -- Steps 4-7 (all formal modelling) will not run. "
          "Install with `pip install statsmodels`.")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Works whether the notebook is run from notebooks/ or the repo root.
_cwd = os.getcwd()
_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == "notebooks" else _cwd
COMPILED_PKL = os.path.join(_ROOT, "data", "compiled_results.pkl")
FLAGGED_CSV  = os.path.join(_ROOT, "data", "compiled_results_flagged.csv")

MAP_CONDITIONS_ORDER = ["baseline", "structural", "temporal_frequency", "temporal_cochange"]
MODEL_ORDER = [
    "mistral/ministral-3b-latest",
    "deepseek/deepseek-v4-flash",
    "fireworks_ai/accounts/fireworks/models/gpt-oss-120b",
    "deepinfra/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B",
]
MODEL_SHORT = {
    "mistral/ministral-3b-latest": "ministral-3b",
    "deepseek/deepseek-v4-flash": "deepseek-flash",
    "fireworks_ai/accounts/fireworks/models/gpt-oss-120b": "gpt-oss-120b",
    "deepinfra/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B": "nemotron-super",
}
TIER_ORDER = ["small", "medium", "large"]

from scipy import stats


def lr_test(model_full, model_reduced):
    """Likelihood-ratio test for two nested statsmodels ML fits.

    statsmodels' `.compare_lr_test()` is only implemented on the
    linear_model results classes (OLS/WLS/GLS) -- it isn't defined on the
    discrete-model results (Logit, NegativeBinomial, ...) used throughout
    Steps 4-7, hence this manual version built from `llf` and `df_model`,
    which both result classes do expose.
    """
    lr_stat = -2 * (model_reduced.llf - model_full.llf)
    df_diff = model_full.df_model - model_reduced.df_model
    p = stats.chi2.sf(lr_stat, df_diff)
    return lr_stat, p, df_diff


## Step 0 — Compile Raw Results

Individual trial records live as one JSON file per `(model, repo, issue, map_condition, rep)`
combination, spread across two directories (this repo's own `results/`, generated before
migrating to a dedicated native-filesystem multi-worker setup, and `study1/results/` on that
native filesystem, where the bulk of the study actually ran, to avoid slow checkout times on
the original filesystem). Compilation is handled by the standalone script
`scripts/compile_results.py` rather than inline here, so it can be re-run independently
whenever new results land, without needing to open this notebook.

The compiler:
- Walks both results directories, restricted to the 4 models in scope for this analysis
- Validates every record against the expected schema, **flagging** (not silently dropping)
  anything malformed, missing required fields, using an unrecognised map type, or duplicated
  across the two source directories
- Joins in `codebase_size` (raw Python LOC) and `tier` from `data/issue_selection_final.csv`,
  since the raw trial records don't carry codebase size themselves
- Keeps only `(model, rep)` pairs with a **complete** trial set (all 45 issues × 4 map
  conditions = 180 trials) — any partial/incomplete set is excluded and reported, not
  silently included at reduced weight
- Saves the result as `data/compiled_results.pkl`, and the flagged/excluded records as
  `data/compiled_results_flagged.csv`

Re-run it from here (safe to re-run any time — it always rebuilds from scratch, doesn't append):


In [ ]:
import subprocess

result = subprocess.run(
    ["python3", os.path.join(_ROOT, "scripts", "compile_results.py")],
    cwd=_ROOT, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("compile_results.py failed -- see output above")


## Step 1 — Load and Validate the Compiled Results

Loads `compiled_results.pkl` and runs sanity checks before any analysis: trials per design cell,
missing data, the overall success/failure class balance, and a check for **"map-independent
failures"** — issues where every map condition (including `baseline`) converged on the *same
wrong* prediction. If a model gets an issue wrong identically regardless of what map (if any) it
was given, that's evidence the map had no influence on that particular failure; lumping these in
with genuinely map-sensitive cases would dilute the analysis. They're flagged here for manual
review (full list in Step 9) rather than silently dropped or silently kept unexamined.

**On the `success` definition**: the compiled table carries `precision`/`recall`/`f1` per trial
(already computed by the harness against the package-scope-and-source-file-filtered ground
truth — see `scripts/source_filter.py` — so files no map or tool could ever surface, like
non-Python files or files outside the resolved package directory, are correctly excluded from
both sides before scoring). `success` is defined here as **`recall == 1.0`**: every ground-truth
file was found, regardless of any extra false-positive files also predicted. This is a
deliberate choice, not the only defensible one — an exact-match definition (`f1 == 1.0`, i.e.
no false positives either) or an any-hit definition (`recall > 0`) would both be reasonable
alternatives for a localisation task; recall-based success was chosen because a developer using
this tool for real would likely tolerate a couple of extra candidate files to check, but would
consider it a failure if the actual fix location was missed entirely. Since `precision`/`recall`/
`f1` are all retained per-trial, this definition can be swapped by changing `df["success"]`
below without recompiling.


In [ ]:
df = pd.read_pickle(COMPILED_PKL)
flagged = pd.read_csv(FLAGGED_CSV) if os.path.exists(FLAGGED_CSV) else pd.DataFrame()

print(f"Loaded {len(df):,} trials")
print(f"Flagged/excluded during compilation: {len(flagged):,}")
print()
print("dtypes:")
print(df.dtypes)


In [ ]:
# Uncomment to redefine success, e.g. to the stricter exact-match version:
# df["success"] = df["f1"] == 1.0

# Trial counts per (model, map_condition) -- should be uniform if the
# design is fully populated (45 issues x n_reps per model, per cell).
counts = df.groupby(["model", "map_condition"]).size().unstack().reindex(columns=MAP_CONDITIONS_ORDER)
counts


In [ ]:
# Missing data check -- should be empty for every analysis-critical column.
critical_cols = ["success", "recall", "precision", "f1", "codebase_size", "turns_used"]
missing = df[critical_cols].isna().sum()
print("Missing values in analysis-critical columns:")
print(missing)
if missing.get("success", 0) > 0:
    print("\nRows with missing 'success' (ground truth was empty after scorable_files()")
    print("filtering -- compute_scores() returns None precision/recall/f1 in that case):")
    display(df[df["success"].isna()][["model", "codebase", "issue_idx", "map_condition", "rep"]])


In [ ]:
print("Overall success rate (recall == 1.0 definition):")
print(df["success"].value_counts(normalize=True, dropna=False))
print()
print(df["success"].value_counts(dropna=False))


In [ ]:
# ── map-independent failures ─────────────────────────────────────────────
# For each (model, codebase, issue_idx, rep), check whether every map
# condition produced the same (wrong) prediction set.
def _pred_key(row):
    return tuple(sorted(row["predicted_files"]))

df["_pred_key"] = df.apply(_pred_key, axis=1)

grp_cols = ["model", "codebase", "issue_idx", "rep"]
map_independent_failures = []
for keys, g in df.groupby(grp_cols):
    if g["map_condition"].nunique() < len(MAP_CONDITIONS_ORDER):
        continue  # incomplete set for this (model, issue, rep) -- skip
    if g["success"].any():
        continue  # at least one condition succeeded -- not map-independent
    if g["_pred_key"].nunique() == 1:
        model, codebase, issue_idx, rep = keys
        map_independent_failures.append({
            "model": model, "codebase": codebase, "issue_idx": issue_idx, "rep": rep,
            "issue_title": g["issue_title"].iloc[0],
            "convergent_prediction": g["predicted_files"].iloc[0],
            "ground_truth": g["ground_truth_files"].iloc[0],
        })

map_independent_df = pd.DataFrame(map_independent_failures)
df = df.drop(columns=["_pred_key"])

print(f"{len(map_independent_df)} (model, issue, rep) combinations are map-independent "
      f"failures -- every condition (baseline included) converged on the identical wrong "
      f"answer.")
print("Flagged here for visibility and listed in full in Step 9; NOT dropped from the main")
print("analysis below by default -- they're still real data points (map_condition legitimately")
print("had no effect for these specific cases), just worth knowing the count and being able to")
print("inspect/exclude them if a reviewer asks.")
if len(map_independent_df):
    display(map_independent_df.head(20))


## Step 2 — Descriptive Summary

Success rate broken down by each main factor in isolation, before any modelling.
`codebase_size` is shown both raw and log-transformed: the primary model (Step 4) uses
`log(codebase_size)` as a predictor, since size spans several orders of magnitude (hundreds to
hundreds of thousands of lines of Python) and a raw-scale linear effect would be dominated
entirely by the largest codebases.


In [ ]:
print("Success rate by map_condition:")
display(df.groupby("map_condition")["success"].agg(["mean", "count"]).reindex(MAP_CONDITIONS_ORDER))

print("\nSuccess rate by model:")
tbl = df.groupby("model")["success"].agg(["mean", "count"]).reindex(MODEL_ORDER)
tbl.index = [MODEL_SHORT[m] for m in tbl.index]
display(tbl)


In [ ]:
df["log_codebase_size"] = np.log(df["codebase_size"])

size_summary = (df[["codebase", "codebase_size", "log_codebase_size", "tier"]]
                 .drop_duplicates(subset="codebase")
                 .sort_values("codebase_size"))
display(size_summary)

print("\nSuccess rate by tier:")
display(df.groupby("tier")["success"].agg(["mean", "count"]).reindex(TIER_ORDER))


In [ ]:
# Full design-cell population check.
cell_counts = (df.groupby(["codebase", "model", "map_condition"]).size()
                 .unstack("map_condition").reindex(columns=MAP_CONDITIONS_ORDER))
n_codebases = cell_counts.shape[0]
expected_cells = n_codebases * len(MODEL_ORDER) * len(MAP_CONDITIONS_ORDER)
print(f"Design cells: {n_codebases} codebases x {len(MODEL_ORDER)} models x "
      f"{len(MAP_CONDITIONS_ORDER)} map conditions = {expected_cells} expected combinations")
print(f"Combinations with zero trials: {(cell_counts.fillna(0) == 0).sum().sum()}")
cell_counts


# Step 3 — Exploratory Visualisation

**These plots are exploratory** — for spotting patterns, outliers, and anomalies before
committing to the formal confirmatory model in Steps 4–8. They are deliberately broad and
numerous; none of them are the "final" reported figures (those are built from the fitted
model's predictions in Step 8, not raw data). Kept in its own section, clearly separated from
the confirmatory analysis below, so a reader doesn't mistake an exploratory chart for a
reported result.


In [ ]:
COLORS = dict(zip(MAP_CONDITIONS_ORDER, sns.color_palette("colorblind", 4))) if HAVE_SEABORN \
         else dict(zip(MAP_CONDITIONS_ORDER, plt.cm.tab10.colors[:4]))


### Success rate by map condition (overall, and by model)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
means = df.groupby("map_condition")["success"].mean().reindex(MAP_CONDITIONS_ORDER)
ax.bar(means.index, means.values, color=[COLORS[c] for c in means.index])
ax.set_ylabel("Success rate (recall == 1.0)")
ax.set_ylim(0, 1)
ax.set_title("Success rate by map condition (all models, all codebases)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(16, 4), sharey=True)
for ax, model in zip(axes, MODEL_ORDER):
    sub = df[df["model"] == model]
    means = sub.groupby("map_condition")["success"].mean().reindex(MAP_CONDITIONS_ORDER)
    ax.bar(means.index, means.values, color=[COLORS[c] for c in means.index])
    ax.set_title(MODEL_SHORT[model], fontsize=10)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Success rate")
fig.suptitle("Success rate by map condition, faceted by model")
plt.tight_layout()
plt.show()


### Success rate by codebase (spot codebase-specific outliers)

In [ ]:
pivot = (df.groupby(["codebase", "map_condition"])["success"].mean()
           .unstack().reindex(columns=MAP_CONDITIONS_ORDER))
# order codebases by size so any size-related pattern is visible left-to-right
pivot = pivot.loc[size_summary.sort_values("codebase_size")["codebase"]]

fig, ax = plt.subplots(figsize=(14, 5))
pivot.plot(kind="bar", ax=ax, color=[COLORS[c] for c in MAP_CONDITIONS_ORDER], width=0.8)
ax.set_ylabel("Success rate")
ax.set_xlabel("Codebase (ordered by size, smallest to largest)")
ax.legend(title="map_condition", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_title("Success rate by codebase, grouped by map condition")
plt.tight_layout()
plt.show()


### Success vs. log(codebase size), by map condition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
rng = np.random.default_rng(0)
for cond in MAP_CONDITIONS_ORDER:
    sub = df[df["map_condition"] == cond].dropna(subset=["success"])
    jitter = rng.uniform(-0.03, 0.03, size=len(sub))
    ax.scatter(sub["log_codebase_size"], sub["success"].astype(float) + jitter,
               alpha=0.15, s=12, color=COLORS[cond], label=None)
    if HAVE_SEABORN:
        try:
            sns.regplot(data=sub, x="log_codebase_size", y="success", logistic=True,
                        ci=95, scatter=False, ax=ax, color=COLORS[cond], label=cond,
                        line_kws={"linewidth": 2})
        except Exception as e:
            print(f"WARNING: logistic trend line for '{cond}' failed ({e}) -- "
                  f"needs statsmodels, which regplot(logistic=True) calls internally.")
    else:
        print("seaborn not installed -- skipping fitted logistic trend lines, showing "
              "jittered points only.")
ax.set_xlabel("log(codebase size, Python LOC)")
ax.set_ylabel("Success (jittered)")
ax.set_title("Success vs. log(codebase size) by map condition, all models pooled")
ax.legend(title="map_condition")
plt.tight_layout()
plt.show()


### Key exploratory chart: success vs. log(size), faceted by model

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(20, 5), sharey=True, sharex=True)
for ax, model in zip(axes, MODEL_ORDER):
    sub_model = df[df["model"] == model]
    for cond in MAP_CONDITIONS_ORDER:
        sub = sub_model[sub_model["map_condition"] == cond].dropna(subset=["success"])
        jitter = rng.uniform(-0.03, 0.03, size=len(sub))
        ax.scatter(sub["log_codebase_size"], sub["success"].astype(float) + jitter,
                   alpha=0.2, s=10, color=COLORS[cond])
        if HAVE_SEABORN:
            try:
                sns.regplot(data=sub, x="log_codebase_size", y="success", logistic=True,
                            ci=95, scatter=False, ax=ax, color=COLORS[cond],
                            line_kws={"linewidth": 2})
            except Exception:
                pass
    ax.set_title(MODEL_SHORT[model], fontsize=11)
    ax.set_xlabel("log(codebase size)")
axes[0].set_ylabel("Success (jittered) / fitted P(success)")
handles = [plt.Line2D([0], [0], color=COLORS[c], lw=2, label=c) for c in MAP_CONDITIONS_ORDER]
fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("Success vs. log(codebase size) x map condition, faceted by model", y=1.12)
plt.tight_layout()
plt.show()


### Turn usage by map condition and model

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(18, 4), sharey=True)
for ax, model in zip(axes, MODEL_ORDER):
    sub = df[df["model"] == model]
    # named turn_stats, not stats -- "stats" would shadow the `from scipy import
    # stats` import in the setup cell, breaking lr_test() (and any other scipy.stats
    # use) in every cell that runs after this one.
    turn_stats = sub.groupby("map_condition")["turns_used"].agg(["mean", "sem"]).reindex(MAP_CONDITIONS_ORDER)
    ax.bar(turn_stats.index, turn_stats["mean"], yerr=turn_stats["sem"], capsize=4,
           color=[COLORS[c] for c in turn_stats.index])
    ax.set_title(MODEL_SHORT[model], fontsize=10)
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Mean turns used (± SEM)")
fig.suptitle("Turns used by map condition, faceted by model")
plt.tight_layout()
plt.show()

### Turn-cap hit rate, by map condition and by codebase size tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

capr = df.groupby("map_condition")["hit_turn_cap"].mean().reindex(MAP_CONDITIONS_ORDER)
axes[0].bar(capr.index, capr.values, color=[COLORS[c] for c in capr.index])
axes[0].set_ylabel("Turn-cap hit rate")
axes[0].set_title("By map condition")
axes[0].tick_params(axis="x", rotation=20)

capr2 = (df.groupby(["tier", "map_condition"])["hit_turn_cap"].mean()
           .unstack().reindex(index=TIER_ORDER, columns=MAP_CONDITIONS_ORDER))
capr2.plot(kind="bar", ax=axes[1], color=[COLORS[c] for c in MAP_CONDITIONS_ORDER])
axes[1].set_title("By codebase-size tier x map condition")
axes[1].set_ylabel("Turn-cap hit rate")
axes[1].legend(title="map_condition", fontsize=8)
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


### Turns used vs. log(size), coloured by success — do forced answers cluster at high turns on large codebases?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for succ, color, label in [(True, "tab:green", "success"), (False, "tab:red", "failure")]:
    sub = df[df["success"] == succ]
    ax.scatter(sub["log_codebase_size"], sub["turns_used"],
               alpha=0.25, s=14, color=color, label=label)
ax.axhline(df["turns_used"].max(), color="grey", linestyle="--", linewidth=1,
           label="approx. turn cap")
ax.set_xlabel("log(codebase size)")
ax.set_ylabel("Turns used")
ax.set_title("Turns used vs. log(codebase size), coloured by outcome")
ax.legend()
plt.tight_layout()
plt.show()


### Heatmap: success rate across (codebase-size bucket x map condition), one panel per model

In [ ]:
n_buckets = 4
df["size_bucket"] = pd.qcut(df["codebase_size"], n_buckets, duplicates="drop")

fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(22, 4.5))
for ax, model in zip(axes, MODEL_ORDER):
    sub = df[df["model"] == model]
    grid = (sub.groupby(["size_bucket", "map_condition"], observed=True)["success"].mean()
               .unstack().reindex(columns=MAP_CONDITIONS_ORDER))
    im = ax.imshow(grid.values, aspect="auto", vmin=0, vmax=1, cmap="RdYlGn")
    ax.set_xticks(range(len(grid.columns)))
    ax.set_xticklabels(grid.columns, rotation=30, ha="right", fontsize=8)
    ax.set_yticks(range(len(grid.index)))
    ax.set_yticklabels([str(iv) for iv in grid.index], fontsize=7)
    ax.set_title(MODEL_SHORT[model], fontsize=10)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            val = grid.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=axes, shrink=0.7, label="success rate")
fig.suptitle("Success rate: codebase-size bucket x map condition, by model")
plt.show()


### Additional exploratory views

In [ ]:
# Cost per trial by map condition and model -- large maps cost more tokens;
# worth seeing whether that cost buys anything.
fig, ax = plt.subplots(figsize=(8, 5))
for model in MODEL_ORDER:
    sub = df[df["model"] == model].groupby("map_condition")["total_cost"].mean().reindex(MAP_CONDITIONS_ORDER)
    ax.plot(sub.index, sub.values, marker="o", label=MODEL_SHORT[model])
ax.set_ylabel("Mean cost per trial (USD)")
ax.set_title("Mean trial cost by map condition, by model")
ax.legend()
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# stop_reason / submission_type distribution -- how trials actually ended.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
df["stop_reason"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("stop_reason distribution (all trials)")
axes[0].tick_params(axis="x", rotation=20)

(df.groupby(["model", "submission_type"]).size().unstack(fill_value=0)
   .reindex(MODEL_ORDER)
   .rename(index=MODEL_SHORT)
   .plot(kind="bar", stacked=True, ax=axes[1]))
axes[1].set_title("submission_type by model")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Precision vs. recall, to see whether any map condition trades one for the
# other systematically (e.g. predicting more files -> higher recall, lower precision).
fig, ax = plt.subplots(figsize=(6, 6))
for cond in MAP_CONDITIONS_ORDER:
    sub = df[df["map_condition"] == cond]
    means = sub[["precision", "recall"]].mean()
    ax.scatter(means["recall"], means["precision"], s=120, color=COLORS[cond], label=cond)
ax.set_xlabel("Mean recall")
ax.set_ylabel("Mean precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Precision vs. recall by map condition (per-condition means)")
ax.legend()
plt.tight_layout()
plt.show()


# Step 4 — Primary Model: Logistic GLMM

**Model**: `success ~ map_condition * log(codebase_size) * model + (1|issue) + (1|codebase)`

Random intercepts for `issue` and `codebase` are **crossed**, not nested (each codebase
contributes 3 issues; `codebase` captures repo-level baseline difficulty/style beyond what
`log(codebase_size)` explains, `issue` captures issue-specific difficulty beyond codebase and
size).

### Library limitation — read before trusting the p-values here

`statsmodels` does not provide a maximum-likelihood binomial GLMM with multiple crossed random
effects. Its only offering in that space, `BinomialBayesMixedGLM`, fits via **variational
Bayes** — an approximation to the posterior, not exact maximum likelihood — and doesn't produce
a deviance suitable for a classical likelihood-ratio test the way `lme4::glmer` + `anova()`
would in R. Two things are done here to work around this rather than silently accept a weaker
test:

1. **`BinomialBayesMixedGLM`** is fit for the full random-effects structure (both `issue` and
   `codebase` intercepts), giving posterior mean/SD coefficient estimates that *do* account for
   the clustering — used for interpretation and as the basis for Step 5's EMMs.
2. A **marginal (non-mixed) logistic regression** (`smf.logit`) is fit alongside it, full vs.
   reduced, specifically to recover a valid **frequentist likelihood-ratio test** for the
   fixed-effects structure (the three-way and two-way interaction terms) via a
   manual `lr_test()` helper (statsmodels' `.compare_lr_test()` is only implemented
   on the linear_model results classes, not discrete models like `Logit`). This
   test is exact for the fixed effects, but — unlike the mixed
   model — treats all 180×N trials as independent, ignoring that trials sharing an `issue` or
   `codebase` are correlated. In practice this means the marginal LRT's p-values are optimistic
   (too small) if there's meaningful issue/codebase-level clustering, which the mixed model's
   random-effect variance estimates (printed below) let you gauge directly.

**If a fully rigorous mixed-model LRT is required for the dissertation**, the recommended path
is exporting `compiled_results.pkl` to CSV and fitting the same formula in R via
`lme4::glmer(success ~ map_condition * log_codebase_size * model + (1|issue) + (1|codebase),
family = binomial)` and `anova(full, reduced)` — noted here rather than silently presenting the
approximation as equivalent.


In [ ]:
if not HAVE_STATSMODELS:
    raise ImportError("statsmodels is required for Steps 4-7. Install with `pip install statsmodels`.")

from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

df_model = df.dropna(subset=["success"]).copy()
df_model["success_int"] = df_model["success"].astype(int)
# Step 7's hit_turn_cap model needs this cast too -- passed as a raw formula endog,
# patsy dummy-codes a bool column into 2 design-matrix columns instead of treating
# it as numeric 0/1, which smf.logit then rejects.
df_model["hit_turn_cap"] = df_model["hit_turn_cap"].astype(int)
df_model["model_short"] = df_model["model"].map(MODEL_SHORT)

REF_MAP_CONDITION = "baseline"
REF_MODEL = MODEL_SHORT["deepseek/deepseek-v4-flash"]  # mid-capability model as reference

print(f"Modelling dataset: {len(df_model)} trials "
      f"({len(df) - len(df_model)} excluded for missing success)")


### Full model (three-way interaction) — `BinomialBayesMixedGLM`

In [ ]:
fixed_formula_full = (
    f"success_int ~ C(map_condition, Treatment('{REF_MAP_CONDITION}')) "
    f"* log_codebase_size "
    f"* C(model_short, Treatment('{REF_MODEL}'))"
)
vc_formulas = {
    "issue":    "0 + C(issue_id)",
    "codebase": "0 + C(codebase)",
}

try:
    bmg_full = BinomialBayesMixedGLM.from_formula(
        fixed_formula_full, vc_formulas, data=df_model)
    bmg_full_result = bmg_full.fit_vb()
    print(bmg_full_result.summary())
except Exception as e:
    bmg_full_result = None
    print(f"BinomialBayesMixedGLM full-model fit failed: {e}")
    print("This can happen with separation (a cell with 0% or 100% success) or a "
          "singular design matrix -- check Step 2's cell-population table for empty or "
          "near-degenerate cells before debugging further.")


### Reduced model (all two-way interactions, no three-way) — `BinomialBayesMixedGLM`

In [ ]:
# Patsy's (a + b + c)**2 expands to all main effects + all pairwise interactions,
# i.e. exactly the full formula minus the three-way term.
fixed_formula_reduced = (
    f"success_int ~ (C(map_condition, Treatment('{REF_MAP_CONDITION}')) "
    f"+ log_codebase_size "
    f"+ C(model_short, Treatment('{REF_MODEL}')))**2"
)

try:
    bmg_reduced = BinomialBayesMixedGLM.from_formula(
        fixed_formula_reduced, vc_formulas, data=df_model)
    bmg_reduced_result = bmg_reduced.fit_vb()
    print(bmg_reduced_result.summary())
except Exception as e:
    bmg_reduced_result = None
    print(f"BinomialBayesMixedGLM reduced-model fit failed: {e}")


### Frequentist LRT for the fixed-effects structure (marginal model, ignores clustering — see caveat above)

In [ ]:
logit_full = smf.logit(fixed_formula_full.replace('success_int', 'success_int'), data=df_model).fit(disp=0)
logit_reduced = smf.logit(fixed_formula_reduced, data=df_model).fit(disp=0)

lr_stat, lr_p, lr_df = lr_test(logit_full, logit_reduced)
print("Likelihood-ratio test: full (3-way interaction) vs. reduced model")
print(f"  LR statistic = {lr_stat:.3f}, df = {lr_df}, p = {lr_p:.4f}")
if lr_p < 0.05:
    print("  -> Three-way interaction IS significant at alpha=0.05: the map effect's")
    print("     dependence on codebase size differs by model. Report the full model.")
else:
    print("  -> Three-way interaction is NOT significant: refit and test the")
    print("     map_condition x model two-way term next (cell below).")


### If the three-way term is non-significant: test `map_condition × model` (collapsing across size)

In [ ]:
# Build this by subtracting just the map_condition:model term from the reduced
# formula's RHS -- patsy expands the (a+b+c)**2 shorthand before applying the
# subtraction, so this correctly drops only that one interaction while keeping
# both other two-way terms and all main effects.
rhs_reduced = fixed_formula_reduced.split(" ~ ")[1]
map_term = f"C(map_condition, Treatment('{REF_MAP_CONDITION}'))"
model_term = f"C(model_short, Treatment('{REF_MODEL}'))"
fixed_formula_no_map_model = f"success_int ~ {rhs_reduced} - {map_term}:{model_term}"

logit_no_map_model = smf.logit(fixed_formula_no_map_model, data=df_model).fit(disp=0)
lr_stat2, lr_p2, lr_df2 = lr_test(logit_reduced, logit_no_map_model)
print("Likelihood-ratio test: reduced model vs. reduced-minus-(map_condition x model)")
print(f"  LR statistic = {lr_stat2:.3f}, df = {lr_df2}, p = {lr_p2:.4f}")
if lr_p2 < 0.05:
    print("  -> map_condition x model interaction IS significant: the map effect is NOT")
    print("     model-agnostic. Report per-model contrasts (Step 5/6), not a pooled one.")
else:
    print("  -> map_condition x model interaction is NOT significant: the map effect")
    print("     appears reasonably consistent across models (supports model-agnosticism).")


### Full vs. reduced model, side by side

In [ ]:
comparison_rows = []
for label, result in [("Full (3-way)", bmg_full_result), ("Reduced (2-way max)", bmg_reduced_result)]:
    if result is None:
        comparison_rows.append({"model": label, "status": "fit failed"})
        continue
    comparison_rows.append({
        "model": label,
        "n_fixed_params": len(result.params) - len(vc_formulas),
        "status": "fit ok",
    })
pd.DataFrame(comparison_rows)


# Step 5 — Estimated Marginal Means & Effect Sizes

**No direct Python equivalent of R's `emmeans` is installed** (`marginaleffects` isn't
available in this environment either — see Setup). EMMs are computed manually here via
prediction at a reference grid, using a `statsmodels` GLM (binomial family, logit link — the
same model as Step 4's marginal logistic regression, refit via the GLM API specifically because
`GLMResults.get_prediction()` gives clean predicted-probability confidence intervals that
`Logit`'s API doesn't expose as directly).

**What's rigorous here and what isn't:**
- Point predictions and CIs **per individual reference-grid cell** (a specific map_condition ×
  size × model combination) are properly computed via the model's own prediction variance — this
  is exactly what `get_prediction()` is for, no approximation involved.
- The **"collapsed across model" summary** (simple average of the per-model predictions at each
  map_condition × size point) reports a point estimate only, **without** a matching confidence
  interval — averaging predictions from a fitted GLM correctly propagating uncertainty requires
  the delta method or a bootstrap, which is exactly what `emmeans`/`marginaleffects` automate and
  neither is available here. Flagged explicitly rather than presenting an interval that would
  understate the true uncertainty. If this collapsed CI is needed for the dissertation, either
  install `marginaleffects` (a pure-Python reimplementation exists as of recent versions) or use
  R's `emmeans` package directly.
- **Pairwise contrasts against baseline** (Step 6) use `t_test()` on an explicit contrast
  vector, which *does* give a statistically valid CI via the delta method — those numbers are
  fully rigorous.


In [ ]:
size_p10, size_p50, size_p90 = df_model["codebase_size"].quantile([0.10, 0.50, 0.90])
log_size_p10, log_size_p50, log_size_p90 = np.log([size_p10, size_p50, size_p90])
print(f"Representative codebase sizes (LOC): p10={size_p10:.0f}, p50={size_p50:.0f}, p90={size_p90:.0f}")

glm_full = smf.glm(fixed_formula_full, data=df_model, family=sm.families.Binomial()).fit()
print(glm_full.summary())


In [ ]:
# Reference grid: every (map_condition, size percentile, model) combination.
grid = pd.DataFrame([
    {"map_condition": mc, "log_codebase_size": ls, "size_label": sl, "model_short": m}
    for mc in MAP_CONDITIONS_ORDER
    for ls, sl in [(log_size_p10, "p10"), (log_size_p50, "p50"), (log_size_p90, "p90")]
    for m in MODEL_SHORT.values()
])

pred = glm_full.get_prediction(grid).summary_frame(alpha=0.05)
grid_pred = pd.concat([grid.reset_index(drop=True), pred.reset_index(drop=True)], axis=1)
grid_pred = grid_pred.rename(columns={"mean": "pred_prob", "mean_ci_lower": "ci_lower", "mean_ci_upper": "ci_upper"})
grid_pred[["map_condition", "size_label", "model_short", "pred_prob", "ci_lower", "ci_upper"]]


### EMMs collapsed across model (point estimate only — see caveat above)

In [ ]:
collapsed = (grid_pred.groupby(["map_condition", "size_label"])["pred_prob"]
             .mean().unstack().reindex(index=MAP_CONDITIONS_ORDER, columns=["p10", "p50", "p90"]))
print("Predicted P(success), averaged across the 4 models (POINT ESTIMATE ONLY, no CI -- see caveat):")
collapsed


### EMMs faceted by model (model-agnosticism check)

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(20, 4.5), sharey=True)
for ax, model in zip(axes, MODEL_ORDER):
    mshort = MODEL_SHORT[model]
    sub = grid_pred[grid_pred["model_short"] == mshort]
    for mc in MAP_CONDITIONS_ORDER:
        row = sub[sub["map_condition"] == mc].set_index("size_label").reindex(["p10", "p50", "p90"])
        x = [log_size_p10, log_size_p50, log_size_p90]
        ax.plot(x, row["pred_prob"], marker="o", color=COLORS[mc], label=mc)
        ax.fill_between(x, row["ci_lower"], row["ci_upper"], color=COLORS[mc], alpha=0.15)
    ax.set_title(mshort, fontsize=10)
    ax.set_xlabel("log(codebase size)")
    ax.set_ylim(0, 1)
axes[0].set_ylabel("Predicted P(success), with 95% CI")
handles = [plt.Line2D([0], [0], color=COLORS[c], lw=2, marker="o", label=c) for c in MAP_CONDITIONS_ORDER]
fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.1))
plt.tight_layout()
plt.show()


### Odds ratios and percentage-point differences: each map condition vs. baseline, at median size

In [ ]:
def contrast_vs_baseline(result, formula_rhs_terms, map_cond, ref_model_short, log_size, design_info):
    """Build a contrast vector for (map_cond vs baseline) at a given log_codebase_size and
    model_short, evaluated via t_test on the fitted GLM's parameters -- gives a proper
    delta-method CI on the log-odds scale, which is then exponentiated for the odds ratio.

    Uses patsy.build_design_matrices() rather than a (nonexistent) DesignInfo.transform()
    method, and .item() rather than float() on the 1-D/2-D arrays t_test() returns for
    .effect/.sd specifically -- numpy 2.x raises on float() of anything but a genuine
    0-dimensional array, which .conf_int() and .pvalue happen to return but .effect/.sd don't.
    """
    point_a = pd.DataFrame([{"map_condition": map_cond, "log_codebase_size": log_size, "model_short": ref_model_short}])
    point_b = pd.DataFrame([{"map_condition": REF_MAP_CONDITION, "log_codebase_size": log_size, "model_short": ref_model_short}])
    x_a = patsy.build_design_matrices([design_info], point_a)[0]
    x_b = patsy.build_design_matrices([design_info], point_b)[0]
    contrast = np.asarray(x_a)[0] - np.asarray(x_b)[0]
    tt = result.t_test(contrast)
    return tt

import patsy
design_info = patsy.dmatrix(fixed_formula_full.split(" ~ ")[1], df_model, return_type="dataframe").design_info

rows = []
for mc in MAP_CONDITIONS_ORDER:
    if mc == REF_MAP_CONDITION:
        continue
    for m in MODEL_SHORT.values():
        tt = contrast_vs_baseline(glm_full, None, mc, m, log_size_p50, design_info)
        log_or, se = tt.effect.item(), tt.sd.item()
        ci_lo, ci_hi = float(tt.conf_int()[0][0]), float(tt.conf_int()[0][1])
        or_pt, or_lo, or_hi = np.exp([log_or, ci_lo, ci_hi])

        p_baseline = collapsed.loc[REF_MAP_CONDITION, "p50"]
        p_cond = collapsed.loc[mc, "p50"]
        rows.append({
            "map_condition": mc, "vs": REF_MAP_CONDITION, "model": m, "at_size": "p50 (median)",
            "odds_ratio": round(or_pt, 3), "OR_ci_lower": round(or_lo, 3), "OR_ci_upper": round(or_hi, 3),
            "p_value": round(float(tt.pvalue), 4),
            "pp_diff_collapsed_estimate": round((p_cond - p_baseline) * 100, 1),
        })

contrasts_df = pd.DataFrame(rows)
contrasts_df


# Step 6 — Planned Pairwise Contrasts

Matched-pairs tests on the same `(model, codebase, issue_idx, rep)` units, exactly the design
McNemar's test and Cochran's Q are built for: comparing correlated binary outcomes measured
under different conditions on the *same* subjects (here, the same trial "subject" tested under
each map condition). Run pooled across all models first (the planned, primary contrast), then
per-model as a secondary breakdown consistent with the model-agnosticism question from Steps 4–5.

- **McNemar's test**, one per `map_condition` vs. `baseline` pair (3 tests)
- **Cochran's Q**, testing all 4 conditions simultaneously on the same matched issue set
- **Holm–Bonferroni correction** applied within the family of 3 McNemar tests (Cochran's Q is a
  single omnibus test, not part of that family)


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar, cochrans_q
from statsmodels.stats.multitest import multipletests

def matched_wide(data):
    """Pivot to one row per (model, codebase, issue_idx, rep), one column per
    map_condition, keeping only rows with a value for all 4 conditions."""
    wide = (data.pivot_table(index=["model", "codebase", "issue_idx", "rep"],
                              columns="map_condition", values="success", aggfunc="first")
                 .dropna())
    return wide[MAP_CONDITIONS_ORDER]

wide_all = matched_wide(df_model)
print(f"{len(wide_all)} matched (model, codebase, issue_idx, rep) units with a complete "
      f"set of all 4 map conditions")


In [ ]:
# ── McNemar, pooled across models ────────────────────────────────────────
mcnemar_rows = []
for cond in MAP_CONDITIONS_ORDER:
    if cond == REF_MAP_CONDITION:
        continue
    a = wide_all[REF_MAP_CONDITION].astype(bool)
    b = wide_all[cond].astype(bool)
    table = pd.crosstab(a, b)
    table = table.reindex(index=[False, True], columns=[False, True], fill_value=0)
    result = mcnemar(table.values, exact=(table.values.sum() < 25))
    n_disagree = table.iloc[0, 1] + table.iloc[1, 0]
    mcnemar_rows.append({
        "comparison": f"{cond} vs {REF_MAP_CONDITION}",
        "n_matched": len(wide_all),
        "n_discordant_pairs": int(n_disagree),
        f"{REF_MAP_CONDITION}_fail_{cond}_succeed": int(table.iloc[0, 1]),
        f"{REF_MAP_CONDITION}_succeed_{cond}_fail": int(table.iloc[1, 0]),
        "statistic": round(float(result.statistic), 3),
        "p_value": float(result.pvalue),
    })

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df["p_holm"] = multipletests(mcnemar_df["p_value"], method="holm")[1]
mcnemar_df["significant_holm_0.05"] = mcnemar_df["p_holm"] < 0.05
mcnemar_df


In [ ]:
# ── Cochran's Q, all 4 conditions, pooled across models ──────────────────
q_result = cochrans_q(wide_all.astype(int).values)
print(f"Cochran's Q = {q_result.statistic:.3f}, df = {len(MAP_CONDITIONS_ORDER)-1}, "
      f"p = {q_result.pvalue:.4f}")
if q_result.pvalue < 0.05:
    print("-> At least one map condition differs from the others on matched issues.")
else:
    print("-> No evidence of a difference among conditions on matched issues (pooled).")


### Per-model breakdown

In [ ]:
per_model_mcnemar = []
for model in MODEL_ORDER:
    wide_m = matched_wide(df_model[df_model["model"] == model])
    for cond in MAP_CONDITIONS_ORDER:
        if cond == REF_MAP_CONDITION:
            continue
        table = pd.crosstab(wide_m[REF_MAP_CONDITION].astype(bool), wide_m[cond].astype(bool))
        table = table.reindex(index=[False, True], columns=[False, True], fill_value=0)
        result = mcnemar(table.values, exact=(table.values.sum() < 25))
        per_model_mcnemar.append({
            "model": MODEL_SHORT[model], "comparison": f"{cond} vs {REF_MAP_CONDITION}",
            "n_matched": len(wide_m), "statistic": round(float(result.statistic), 3),
            "p_value": float(result.pvalue),
        })

per_model_df = pd.DataFrame(per_model_mcnemar)
per_model_df["p_holm"] = multipletests(per_model_df["p_value"], method="holm")[1]
per_model_df


# Step 7 — Secondary Outcomes

Same fixed-effects structure as the primary model, applied to two secondary outcomes:
**turn count** (how much exploration a trial needed) and **turn-cap hit rate** (how often a
trial ran out of budget without voluntarily converging). Both use the same marginal-model
approach and caveat as Step 4 (no crossed-random-effects MLE available in `statsmodels`).


### Turn count — negative binomial regression

In [ ]:
nb_full = smf.negativebinomial(
    fixed_formula_full.replace("success_int", "turns_used"), data=df_model
).fit(disp=0)
nb_reduced = smf.negativebinomial(
    fixed_formula_reduced.replace("success_int", "turns_used"), data=df_model
).fit(disp=0)
nb_lr_stat, nb_lr_p, nb_lr_df = lr_test(nb_full, nb_reduced)
print(f"Negative binomial: three-way interaction LRT: stat={nb_lr_stat:.3f}, "
      f"df={nb_lr_df}, p={nb_lr_p:.4f}")
print()
print(nb_full.summary())


### Turn-cap hit rate — logistic regression

In [ ]:
try:
    cap_full = smf.logit(
        fixed_formula_full.replace("success_int", "hit_turn_cap"), data=df_model
    ).fit(disp=0)
except Exception as e:
    cap_full = None
    print(f"hit_turn_cap full-model fit failed: {e}")
    print("Likely separation -- several (model, map_condition) cells have a 0% hit_turn_cap")
    print("rate (ministral-3b and gpt-oss-120b rarely exhaust the turn budget at all), which")
    print("the full 3-way interaction model can't estimate a finite coefficient for.")

try:
    cap_reduced = smf.logit(
        fixed_formula_reduced.replace("success_int", "hit_turn_cap"), data=df_model
    ).fit(disp=0)
except Exception as e:
    cap_reduced = None
    print(f"hit_turn_cap reduced-model fit failed: {e}")

if cap_full is not None and cap_reduced is not None:
    cap_lr_stat, cap_lr_p, cap_lr_df = lr_test(cap_full, cap_reduced)
    print(f"hit_turn_cap: three-way interaction LRT: stat={cap_lr_stat:.3f}, "
          f"df={cap_lr_df}, p={cap_lr_p:.4f}")
    print()
    print(cap_full.summary())
else:
    print("Skipping the hit_turn_cap LRT and summary -- see fit failure(s) above.")

### Voluntary vs. forced submission × correct vs. incorrect

In [ ]:
df_model["submission_kind"] = np.where(
    df_model["submission_type"] == "submit_answer", "voluntary", "forced/other")

for model in MODEL_ORDER:
    print(f"\n=== {MODEL_SHORT[model]} ===")
    sub = df_model[df_model["model"] == model]
    for cond in MAP_CONDITIONS_ORDER:
        ct = pd.crosstab(sub[sub["map_condition"] == cond]["submission_kind"],
                          sub[sub["map_condition"] == cond]["success"])
        print(f"\n{cond}:")
        print(ct)


# Step 8 — Confirmatory Figures (Report-Quality)

Unlike Step 3's exploratory plots, these are built from the **fitted model's predictions**
(Step 5's reference-grid predictions), not raw data — the intended headline figures for
communicating the map × size × model result.


### Headline figure: predicted P(success) vs. log(size), one panel per model

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(20, 5), sharey=True)
size_grid_fine = np.linspace(df_model["log_codebase_size"].min(), df_model["log_codebase_size"].max(), 30)

for ax, model in zip(axes, MODEL_ORDER):
    mshort = MODEL_SHORT[model]
    fine_grid = pd.DataFrame([
        {"map_condition": mc, "log_codebase_size": ls, "model_short": mshort}
        for mc in MAP_CONDITIONS_ORDER for ls in size_grid_fine
    ])
    fine_pred = glm_full.get_prediction(fine_grid).summary_frame(alpha=0.05)
    fine_grid = pd.concat([fine_grid.reset_index(drop=True), fine_pred.reset_index(drop=True)], axis=1)

    for mc in MAP_CONDITIONS_ORDER:
        s = fine_grid[fine_grid["map_condition"] == mc]
        ax.plot(s["log_codebase_size"], s["mean"], color=COLORS[mc], label=mc, linewidth=2)
        ax.fill_between(s["log_codebase_size"], s["mean_ci_lower"], s["mean_ci_upper"],
                         color=COLORS[mc], alpha=0.15)
    ax.set_title(mshort, fontsize=12)
    ax.set_xlabel("log(codebase size)")
    ax.set_ylim(0, 1)

axes[0].set_ylabel("Predicted P(success)")
handles = [plt.Line2D([0], [0], color=COLORS[c], lw=2, label=c) for c in MAP_CONDITIONS_ORDER]
fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("Predicted P(success) vs. codebase size, by map condition and model", y=1.14)
plt.tight_layout()
plt.savefig(os.path.join(_ROOT, "notebooks", "fig_headline_by_model.png"), dpi=150, bbox_inches="tight")
plt.show()


### Simplified headline: collapsed across model

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
collapsed_fine = (pd.DataFrame([
        {"map_condition": mc, "log_codebase_size": ls, "model_short": m}
        for mc in MAP_CONDITIONS_ORDER for ls in size_grid_fine for m in MODEL_SHORT.values()
    ])
    .assign(pred=lambda d: glm_full.predict(d))
    .groupby(["map_condition", "log_codebase_size"])["pred"].mean().reset_index())

for mc in MAP_CONDITIONS_ORDER:
    s = collapsed_fine[collapsed_fine["map_condition"] == mc]
    ax.plot(s["log_codebase_size"], s["pred"], color=COLORS[mc], label=mc, linewidth=2)
ax.set_xlabel("log(codebase size)")
ax.set_ylabel("Predicted P(success), averaged across models (no CI -- see Step 5 caveat)")
ax.set_ylim(0, 1)
ax.set_title("Predicted P(success) vs. codebase size, by map condition (collapsed across model)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(_ROOT, "notebooks", "fig_headline_collapsed.png"), dpi=150, bbox_inches="tight")
plt.show()


### Secondary confirmatory figures: turn count and turn-cap rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for mc in MAP_CONDITIONS_ORDER:
    fine_grid = pd.DataFrame([
        {"map_condition": mc, "log_codebase_size": ls, "model_short": m}
        for ls in size_grid_fine for m in MODEL_SHORT.values()
    ])
    fine_grid["pred_turns"] = nb_full.predict(fine_grid)
    agg_cols = ["pred_turns"]
    if cap_full is not None:
        fine_grid["pred_cap"] = cap_full.predict(fine_grid)
        agg_cols.append("pred_cap")
    agg = fine_grid.groupby("log_codebase_size")[agg_cols].mean().reset_index()
    axes[0].plot(agg["log_codebase_size"], agg["pred_turns"], color=COLORS[mc], label=mc, linewidth=2)
    if cap_full is not None:
        axes[1].plot(agg["log_codebase_size"], agg["pred_cap"], color=COLORS[mc], label=mc, linewidth=2)

axes[0].set_title("Predicted turns used vs. codebase size")
axes[0].set_xlabel("log(codebase size)")
axes[0].set_ylabel("Predicted turns used")
if cap_full is not None:
    axes[1].set_title("Predicted P(hit turn cap) vs. codebase size")
    axes[1].set_xlabel("log(codebase size)")
    axes[1].set_ylabel("Predicted P(hit turn cap)")
    axes[1].set_ylim(0, 1)
else:
    axes[1].text(0.5, 0.5, "hit_turn_cap model fit failed\n(separation -- see Step 7)",
                 ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_axis_off()
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

# Step 9 — Diagnostic Appendix

## Ground-truth file survival rate per map condition

Structural/temporal maps are token-budget-capped (55k tokens) and files are pruned
least-frequently-edited first when a map would otherwise exceed that budget (see
`scripts/generate_pruned_maps.py` / `generate_pruned_aux_maps.py`). If a ground-truth file gets
pruned away, that trial has **zero chance of success under that map condition specifically**,
regardless of model quality — a structural confound distinct from "the map didn't help". This
section reports, per codebase and per map condition, what fraction of issues kept *all* their
ground-truth files after pruning, using the audit already run this project
(`data/pruned_ground_truth_check_55k.csv` for `structural`, `data/aux_pruned_ground_truth_check_55k.csv`
for the two temporal conditions). `baseline` has no map and therefore no pruning — trivially
100% survival, included for completeness.


In [ ]:
struct_check = pd.read_csv(os.path.join(_ROOT, "data", "pruned_ground_truth_check_55k.csv"))
aux_check = pd.read_csv(os.path.join(_ROOT, "data", "aux_pruned_ground_truth_check_55k.csv"))

survival_rows = []
for _, row in struct_check.iterrows():
    survival_rows.append({
        "codebase": row["repo"], "issue_idx": row["issue_idx"],
        "map_condition": "structural", "gt_survived": not row["any_gt_pruned"],
    })
for _, row in aux_check.iterrows():
    survival_rows.append({
        "codebase": row["repo"], "issue_idx": row["issue_idx"],
        "map_condition": "temporal_frequency", "gt_survived": not row["any_gt_pruned_freq"],
    })
    survival_rows.append({
        "codebase": row["repo"], "issue_idx": row["issue_idx"],
        "map_condition": "temporal_cochange", "gt_survived": not row["any_gt_pruned_cochange"],
    })
for codebase, issue_idx in df[["codebase", "issue_idx"]].drop_duplicates().itertuples(index=False):
    survival_rows.append({
        "codebase": codebase, "issue_idx": issue_idx,
        "map_condition": "baseline", "gt_survived": True,
    })

survival_df = pd.DataFrame(survival_rows).drop_duplicates(subset=["codebase", "issue_idx", "map_condition"])

print("Overall GT survival rate by map condition:")
display(survival_df.groupby("map_condition")["gt_survived"].mean().reindex(MAP_CONDITIONS_ORDER))

print("\nPer-codebase survival rate (flagging any codebase below 90% for any condition):")
per_codebase_survival = (survival_df.groupby(["codebase", "map_condition"])["gt_survived"].mean()
                          .unstack().reindex(columns=MAP_CONDITIONS_ORDER))
display(per_codebase_survival)

poor_survival = per_codebase_survival[(per_codebase_survival < 0.9).any(axis=1)]
if len(poor_survival):
    print("\n*** CAVEAT: the following codebases have <90% ground-truth survival for at least")
    print("one map condition -- results for these codebases under that condition should be")
    print("interpreted with this structural confound in mind, not purely as a map-quality signal:")
    display(poor_survival)


## Map-independent failures (full list)

In [ ]:
print(f"{len(map_independent_df)} (model, issue, rep) combinations where every map condition")
print("(baseline included) converged on the identical wrong prediction -- listed in full below")
print("for manual review. These are cases the map genuinely could not have influenced.")
pd.set_option("display.max_colwidth", 200)
map_independent_df.sort_values(["model", "codebase", "issue_idx"])
